# DEMONSTRATION DU PROJET DE DETECTION D'INTRUSION

In [2]:
import sys
import os
import pandas as pd
import numpy as np
sys.path.append(os.path.abspath(".."))
# Recharger automatiquement un module dès que tu modifies son fichier .py
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


---

## Chargement des données necessaires pour la demonstration (apres Split 70/15/15)

In [3]:
# 📁 Chargement des données UNSW_NB15 (avant équilibrage)
X_train_u = pd.read_csv("../data/splits/unsw/X_train.csv")
y_train_u = pd.read_csv("../data/splits/unsw/y_train.csv")
X_test_u  = pd.read_csv("../data/splits/unsw/X_test.csv")
y_test_u  = pd.read_csv("../data/splits/unsw/y_test.csv") 


---

## Entrainement du modèle RF sur UNSW 

In [4]:
from fonctions.model_rf import train_rf, save_model, load_model
from fonctions.evaluate import evaluate_model


In [5]:
rf_model_noeq_u = train_rf(X_train_u, y_train_u)

#Sauvegarde du modele
model_path = "../models/RF/UNSW/no_balancing/random_forest_baseline.pkl"
save_model(rf_model_noeq_u, model_path)

metrics_dt_u, y_pred_dt_u = evaluate_model(
    rf_model_noeq_u,
    X_test_u,
    y_test_u,
    "../results/RF/UNSW/no_balancing"
) 

🔥 Entraînement du Random Forest...
✔️ Modèle RF entraîné avec succès.
💾 Modèle sauvegardé sous : ../models/RF/UNSW/no_balancing/random_forest_baseline.pkl
🤖 Prédiction du modèle...
📄 metrics.csv généré.
📄 classification_report.csv + .txt générés.
🖼️ confusion_matrix.png générée.
🖼️ feature_importance.png générée.
🎉 Évaluation terminée !


---

## Attaque du modèle par feature perturbation

In [6]:
from fonctions.feature_attack import get_immutable_mask_unsw_RF, feature_perturbation_rf_universal
from fonctions.model_rf import load_model

In [7]:
immutable_mask_rf_u = get_immutable_mask_unsw_RF(X_train_u.columns)

rf_model_noeq_u = load_model("../models/RF/UNSW/no_balancing/random_forest_baseline.pkl")

# y_test doit être 1D
y_test_u_1d = y_test_u.values.reshape(-1)

X_adv_unsw, min_u_eff, max_u_eff = feature_perturbation_rf_universal(
    model=rf_model_noeq_u,
    X_test_real=X_test_u.values,
    y_test=y_test_u.values.reshape(-1),
    min_vals=X_train_u.values.min(axis=0),
    max_vals=X_train_u.values.max(axis=0),
    immutable_mask=immutable_mask_rf_u,
    feature_names=X_test_u.columns,
    max_ratio=0.02,
    n_features=5,
    seed=42,
    output_dir="../attacks/feature_perturbation/RF/UNSW/no_balancing"
)

📁 Modèle chargé depuis : ../models/RF/UNSW/no_balancing/random_forest_baseline.pkl
📌 Dataset détecté : UNSW
🔍 Features modifiées : [6, 19, 3, 18, 0]
📁 X_adv sauvegardé dans : ..\attacks\feature_perturbation\RF\UNSW\no_balancing


---

## Verification du realiste de l'attaque avec la plausibilité

In [8]:
from fonctions.plausibility import check_plausibility_RF

In [9]:
stats_u, verdict_u = check_plausibility_RF(
    X_clean=X_test_u.values,
    X_adv=X_adv_unsw,
    min_vals=min_u_eff,
    max_vals=max_u_eff,
    output_dir="../attacks/feature_perturbation/RF/UNSW/no_balancing"
)

📌 Dataset détecté pour plausibility : UNSW
✔️ Seuil L2 utilisé    : 20
✔️ Seuil L∞ utilisé    : 600

🔍 Résultats du contrôle de plausibilité :
{'l2_distance_mean': 3.74431088966384, 'l_inf_max': 503.8928457519515, 'nb_negative': 0, 'nb_nan': 0, 'nb_above_max': 0}

📢 VERDICT DE PLAUSIBILITÉ
---------------------------
✅ L’attaque est RÉALISTE.

📁 Rapport sauvegardé dans : ..\attacks\feature_perturbation\RF\UNSW\no_balancing


---

## Sauvegarde des données d'attaque dans un seul dossier 

In [10]:
from fonctions.utils_adversarial import save_adversarial_dataset

save_adversarial_dataset(
    X_adv_path="../attacks/feature_perturbation/RF/UNSW/no_balancing/X_adv_real_FP.npy",
    X_test=X_test_u,
    y_test= y_test_u.iloc[:, 0],
    attack_name="feature_perturbation",
    model_name="RF",
    dataset_name="UNSW",
    balancing="no_balancing",
    plausibility_path= "../attacks/feature_perturbation/RF/UNSW/no_balancing/plausibility.json",
    epsilon=None
    )

print("✔ Terminé\n")


📥 Loaded X_adv (NPY) from ../attacks/feature_perturbation/RF/UNSW/no_balancing/X_adv_real_FP.npy
✅ Dataset adversarial sauvegardé dans : ../data/adversarial\UNSW\feature_perturbation\RF\no_balancing
✔ Terminé



---

## Evaluation des resultats avant attaque et après attaque

### Tableau comparatif des metrics

In [11]:
from fonctions.evaluation_adv import load_labels_generic, evaluate_us16_unsw_and_save_fp_fn

In [12]:
# Pour UNSW feature perturbation avec donnes non equilibres
metrics, path = evaluate_us16_unsw_and_save_fp_fn(
    model=load_model("../models/RF/UNSW/no_balancing/random_forest_baseline.pkl"),
    X_clean=pd.read_csv("../data/splits/unsw/X_test.csv").values,
    y_clean=load_labels_generic("../data/splits/unsw/y_test.csv"),
    X_adv=pd.read_csv("../data/adversarial/UNSW/feature_perturbation/RF/no_balancing/X_adv.csv").values,
    y_true=load_labels_generic("../data/adversarial/UNSW/feature_perturbation/RF/no_balancing/y_true.csv"),
    save_dir="../compare_after_attack/UNSW/FP/no_balancing/"
)

📁 Modèle chargé depuis : ../models/RF/UNSW/no_balancing/random_forest_baseline.pkl
📁 Tableau UNSW FP+FN sauvegardé dans : ../compare_after_attack/UNSW/FP/no_balancing/tableau_comparatif_unsw.md


### Graphique sur les features les plus touchés par l'attaque 

In [13]:
from fonctions.feature_impact import compute_feature_impact, plot_feature_impact_heatmap

In [14]:
feature_names = X_train_u.columns.tolist()
model = load_model("../models/RF/UNSW/no_balancing/random_forest_baseline.pkl")
X_test = pd.read_csv("../data/splits/unsw/X_test.csv").values
y_test = pd.read_csv("../data/splits/unsw/y_test.csv").values.ravel()
X_adv = np.load("../attacks/feature_perturbation/RF/UNSW/no_balancing/X_adv_real_FP.npy")
save_dir = "../results/feature_impact_FP/UNSW/no_balancing/"

impact_dict, sorted_features = compute_feature_impact(
    model=model,
    X_test=X_test,
    y_test=y_test,
    X_adv=X_adv,
    feature_names=feature_names,
    save_dir=save_dir
)

plot_feature_impact_heatmap(sorted_features, save_dir, top_k=20)


📁 Modèle chargé depuis : ../models/RF/UNSW/no_balancing/random_forest_baseline.pkl

Accuracy avant attaque : 0.9385
💾 feature_impact.csv sauvegardé dans ../results/feature_impact_FP/UNSW/no_balancing/feature_impact.csv
💾 top20_features.json sauvegardé dans ../results/feature_impact_FP/UNSW/no_balancing/top20_features.json
💾 heatmap_top20.png sauvegardée dans ../results/feature_impact_FP/UNSW/no_balancing/heatmap_top20.png


---

## Defense contre les feature perturbés

In [15]:
from fonctions.defense_feature_perturbation_rf import defense_feature_perturbation_rf

In [16]:
# ------------------------------------------------------------
# UNSW — no_balancing
# ------------------------------------------------------------
defense_feature_perturbation_rf(
    X_train_path="../data/splits/unsw/X_train.csv",
    y_train_path="../data/splits/unsw/y_train.csv",
    baseline_model_path="../models/RF/UNSW/no_balancing/random_forest_baseline.pkl",
    save_model_path="../models/RF/UNSW/no_balancing/rf_defended_FP.pkl",
    dataset="UNSW"
)


=== DEFENSE FEATURE PERTURBATION RF (UNSW) ===
📁 Modèle chargé depuis : ../models/RF/UNSW/no_balancing/random_forest_baseline.pkl
📌 Dataset détecté : UNSW
🔍 Features modifiées : [6, 19, 3, 18, 0]
📁 X_adv sauvegardé dans : ..\models\RF\UNSW\no_balancing\FP_attack_train
📌 Dataset détecté pour plausibility : UNSW
✔️ Seuil L2 utilisé    : 20
✔️ Seuil L∞ utilisé    : 600

🔍 Résultats du contrôle de plausibilité :
{'l2_distance_mean': 3.7296221842669977, 'l_inf_max': 2.9999892182974888, 'nb_negative': 0, 'nb_nan': 0, 'nb_above_max': 0}

📢 VERDICT DE PLAUSIBILITÉ
---------------------------
✅ L’attaque est RÉALISTE.

📁 Rapport sauvegardé dans : ..\models\RF\UNSW\no_balancing\FP_attack_train
Plausibility verdict : True
✔ Modèle défendu sauvegardé dans : ../models/RF/UNSW/no_balancing/rf_defended_FP.pkl


RandomForestClassifier(n_estimators=200, n_jobs=-1, random_state=42)

In [17]:
from fonctions.evaluate_defense_fp import evaluate_fp_pipeline_unsw
df_results = evaluate_fp_pipeline_unsw(
    baseline_model_path="../models/RF/UNSW/no_balancing/random_forest_baseline.pkl",
    defended_model_path="../models/RF/UNSW/no_balancing/rf_defended_FP.pkl",
    X_test_path="../data/splits/UNSW/X_test.csv",
    y_test_path="../data/splits/UNSW/y_test.csv",
    X_adv_path="../attacks/feature_perturbation/RF/UNSW/no_balancing/X_adv_real_FP.npy",
    save_dir="../compare_after_defense/feature_defense/UNSW/no_balancing/"
)

df_results


✔ Tableau UNSW sauvegardé dans : ../compare_after_defense/feature_defense/UNSW/no_balancing/comparison_feature_defense_unsw.csv



,Model,accuracy,recall,f1,false_positives,false_negatives
0,Baseline – Clean,0.938509,0.950673,0.952036,1136,1206
1,Baseline – FP,0.642792,0.999714,0.782282,13598,7
2,Defended – FP,0.909654,0.953659,0.93128,2308,1133
3,Delta (Def FP - Base FP),+0.2669,-0.0461,+0.1490,-11290.0000,+1126.0000


---

## Detection d'attaque 

In [18]:
from fonctions.detector_isoforest import run_US21

In [19]:
# Feature perturbation UNSW donnes non equilibres 

X_adv   = np.load("../attacks/feature_perturbation/RF/UNSW/no_balancing/X_adv_real_FP.npy")
detector, metrics = run_US21(
    X_train=X_train_u.values,
    X_clean=X_test_u.values,
    X_adv=X_adv,
    output_dir="../defenses/detector_FP/UNSW/no_balancing/"
)


📁 Résultats sauvegardés dans : ../defenses/detector_FP/UNSW/no_balancing/US21_results.json


---

## Evaluation des metric passifs 

In [20]:
from fonctions.non_functional_metrics import save_non_functional_results, evaluate_non_functional
from joblib import load

In [21]:
# ===============================
#  RF Baseline
# ===============================
results_rf = evaluate_non_functional(
    model=load_model("../models/RF/UNSW/no_balancing/random_forest_baseline.pkl"),
    model_path="../models/RF/UNSW/no_balancing/random_forest_baseline.pkl",
    X_test=X_test_u.values,
    is_mlp=False
)
save_non_functional_results("RF_UNSW_baseline", results_rf)


# ===============================
#  RF Défendu FP
# ===============================
results_rf_def = evaluate_non_functional(
    model=load("../models/RF/UNSW/no_balancing/rf_defended_FP.pkl", mmap_mode="r"),
    model_path="../models/RF/UNSW/no_balancing/rf_defended_FP.pkl",
    X_test=X_test_u.values,
    is_mlp=False
)
save_non_functional_results("RF_UNSW_defended_FP", results_rf_def)

print(results_rf)
print(results_rf_def)

📁 Modèle chargé depuis : ../models/RF/UNSW/no_balancing/random_forest_baseline.pkl
📁 Résultats sauvegardés dans : ../results/non_functional/RF_UNSW_baseline_metrics.json
📁 Résultats sauvegardés dans : ../results/non_functional/RF_UNSW_defended_FP_metrics.json
{'latence_ms': 39.4634, 'latence_std': 3.6062, 'taille_modele_mb': 124.9285, 'memoire_ram_mb': 1904.4922}
{'latence_ms': 64.9247, 'latence_std': 4.2112, 'taille_modele_mb': 673.6442, 'memoire_ram_mb': 2438.5156}
